In [14]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import colorsys

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader

In [15]:
FAMILY_NAME = "modulo_addition_1layer"

# Variants that received extended training (~35K epochs)
EXTENDED_MODELS = [
    (59,  485, 598, "p59/s485/ds598"),
    (89,  999, 598, "p89 smooth-diverge"),
    (101, 485, 42,  "p101/s485/ds42"),
    (101, 999, 598, "p101 open-loop"),
    (113, 999, 42,  "p113/s999/ds42"),
]

# Epoch at which we fit the PCA basis for before-and-after
EPOCH_BASIS = 25000

family = load_family(FAMILY_NAME)

## Core helpers

**Trajectory PCA** treats each epoch's flattened W_in as a point in weight space.
PCA across epochs reveals the principal directions of *change* during training.

Unlike the per-epoch scatter in `parameter_space_pca.ipynb` (where neurons are points),
here **epochs are points** — one per snapshot.

In [16]:
def load_win_trajectory(variant, epoch_limit=None):
    """Load W_in at all available checkpoints and return a stacked trajectory matrix.

    W_in is (d_model, d_mlp); each epoch is flattened to a single row.

    Args:
        variant:      Variant object
        epoch_limit:  if set, only load epochs <= this value

    Returns:
        epochs:       list[int]                   — epoch indices used
        traj_matrix:  np.ndarray (n_epochs, d_model*d_mlp)
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    available = sorted(loader.get_epochs("parameter_snapshot"))
    if epoch_limit is not None:
        available = [e for e in available if e <= epoch_limit]

    rows = []
    for epoch in available:
        snap = loader.load_epoch("parameter_snapshot", epoch)
        rows.append(snap["W_in"].flatten())

    return available, np.stack(rows)   # (n_epochs, d_model*d_mlp)


def fit_trajectory_pca(traj_matrix, n_components=3):
    """PCA over a trajectory matrix where rows are epochs.

    Returns:
        coords:       (n_epochs, n_components)
        basis:        (n_components, d_params)  — PC directions
        center:       (d_params,)               — mean parameter vector
        var_explained (n_components,)
    """
    center = traj_matrix.mean(axis=0)
    X = traj_matrix - center
    _, S, Vt = np.linalg.svd(X, full_matrices=False)
    basis = Vt[:n_components]                     # (n_components, d_params)
    coords = X @ basis.T                          # (n_epochs, n_components)
    var_explained = (S**2 / (S**2).sum())[:n_components]
    return coords, basis, center, var_explained


def project_trajectory(traj_matrix, basis, center):
    """Project an arbitrary trajectory matrix through a pre-fit PCA basis."""
    X = traj_matrix - center
    return X @ basis.T                            # (n_epochs, n_components)


def _epoch_colorscale(epochs):
    """Map epoch indices to a sequential blue→red colorscale value (0–1)."""
    lo, hi = min(epochs), max(epochs)
    return [(e - lo) / max(hi - lo, 1) for e in epochs]

---
## 1. Before-and-after Parameter Trajectory PCA

For models that received extended training beyond epoch 25K:

- **Approach A** — basis fit on epochs 0–25K; only those epochs projected and displayed.  
  Reproduces the "original" view: what did training look like before the extension?

- **Approach B** — same 25K basis; all epochs 0–35K projected.  
  If the model *returns* to an earlier weight state, the extended trajectory
  overlaps the earlier arc in this fixed-basis space.

Each panel is a 2D scatter of **PC2 vs PC1**, colored by epoch (blue → red).  
The trajectory becomes a path through the principal-change plane.

In [17]:
def plot_before_after(variant, label, epoch_basis=EPOCH_BASIS) -> go.Figure:
    """2x2 grid: PC1vPC2 and PC2vPC3 for Approach A (original) and Approach B (extended).

    Approach A: basis from first `epoch_basis` epochs; only those epochs shown.
    Approach B: same basis; all available epochs projected.
    The PC2vPC3 panel is the loop diagnostic — shows whether the trajectory closes.
    """
    epochs_a, traj_a = load_win_trajectory(variant, epoch_limit=epoch_basis)
    coords_a, basis, center, var_exp = fit_trajectory_pca(traj_a)

    epochs_all, traj_all = load_win_trajectory(variant)
    coords_all = project_trajectory(traj_all, basis, center)

    col_a   = _epoch_colorscale(epochs_a)
    col_all = _epoch_colorscale(epochs_all)

    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]
    prime = int(variant.model_config["prime"])

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f"A: PC1 vs PC2  (0–{epoch_basis//1000}K basis + view)",
            f"B: PC1 vs PC2  (full projection)",
            f"A: PC2 vs PC3  (loop diagnostic)",
            f"B: PC2 vs PC3  (loop diagnostic, full projection)",
        ],
        horizontal_spacing=0.12,
        vertical_spacing=0.14,
    )

    pre_mask  = [e <= epoch_basis for e in epochs_all]
    post_mask = [e >  epoch_basis for e in epochs_all]

    # (x_component, y_component, panel_row)
    planes = [
        (0, 1, 1),   # PC1 vs PC2
        (1, 2, 2),   # PC2 vs PC3
    ]

    for x_comp, y_comp, row in planes:
        x_lbl = axis_labels[x_comp]
        y_lbl = axis_labels[y_comp]
        hover_tmpl = (
            f"epoch %{{customdata}}<br>"
            f"PC{x_comp+1}=%{{x:.3f}} PC{y_comp+1}=%{{y:.3f}}<extra></extra>"
        )

        # Approach A
        fig.add_trace(
            go.Scatter(
                x=coords_a[:, x_comp].tolist(),
                y=coords_a[:, y_comp].tolist(),
                mode="lines+markers",
                marker=dict(size=5, color=col_a, colorscale="RdYlBu_r", cmin=0, cmax=1, showscale=False),
                line=dict(width=1, color="rgba(100,100,100,0.3)"),
                customdata=epochs_a,
                hovertemplate=hover_tmpl,
                showlegend=False,
            ),
            row=row, col=1,
        )
        fig.update_xaxes(title_text=x_lbl, row=row, col=1)
        fig.update_yaxes(title_text=y_lbl, row=row, col=1)

        # Approach B — two segments so pre/post can be styled independently
        def _segment(mask, name, opacity, show_legend, show_colorbar):
            idx = [i for i, m in enumerate(mask) if m]
            if not idx:
                return
            fig.add_trace(
                go.Scatter(
                    x=coords_all[idx, x_comp].tolist(),
                    y=coords_all[idx, y_comp].tolist(),
                    mode="lines+markers",
                    name=name,
                    marker=dict(
                        size=5,
                        color=[col_all[i] for i in idx],
                        colorscale="RdYlBu_r",
                        cmin=0, cmax=1,
                        showscale=show_colorbar,
                        colorbar=dict(title="epoch", len=0.4, x=1.02, y=0.2) if show_colorbar else None,
                        opacity=opacity,
                    ),
                    line=dict(width=1, color="rgba(100,100,100,0.3)"),
                    customdata=[epochs_all[i] for i in idx],
                    hovertemplate=hover_tmpl,
                    showlegend=show_legend,
                ),
                row=row, col=2,
            )
        _segment(pre_mask,  "0–25K",    0.6,  show_legend=(row == 1), show_colorbar=False)
        _segment(post_mask, "extended", 0.95, show_legend=(row == 1), show_colorbar=(row == 2))
        fig.update_xaxes(title_text=x_lbl, row=row, col=2)
        fig.update_yaxes(title_text=y_lbl, row=row, col=2)

    fig.update_layout(
        title=f"{label} | W_in trajectory PCA (basis at epoch ≤{epoch_basis:,})",
        template="plotly_white",
        height=900,
        legend=dict(x=0.5, y=-0.05, orientation="h", xanchor="center"),
    )
    return fig

In [18]:
# Run all extended-training variants
for prime, model_seed, data_seed, label in EXTENDED_MODELS:
    v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)
    fig = plot_before_after(v, label=label)
    print(f"{label}: {fig.data[0].x.__class__.__name__}")
    fig.show()

p59/s485/ds598: tuple


p89 smooth-diverge: tuple


p101/s485/ds42: tuple


p101 open-loop: tuple


p113/s999/ds42: tuple


### PC1 / PC2 / PC3 vs epoch — time-series view

The scatter above shows the *shape* of the trajectory; this panel shows the
individual PCs vs training time so inflection timing is directly readable.

In [19]:
def plot_trajectory_timeseries(variant, label, epoch_basis=EPOCH_BASIS) -> go.Figure:
    """PC1/PC2/PC3 vs epoch for full trajectory, basis fit at epoch_basis.

    Vertical dashed line marks the basis epoch; shaded region = extended training.
    """
    epochs_b, traj_b = load_win_trajectory(variant, epoch_limit=epoch_basis)
    _, basis, center, var_exp = fit_trajectory_pca(traj_b)

    epochs_all, traj_all = load_win_trajectory(variant)
    coords_all = project_trajectory(traj_all, basis, center)

    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]
    prime = int(variant.model_config["prime"])

    n_comp = coords_all.shape[1]
    fig = make_subplots(
        rows=n_comp, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=axis_labels,
    )

    pre_mask  = [e <= epoch_basis for e in epochs_all]
    post_mask = [e >  epoch_basis for e in epochs_all]

    colors = ["steelblue", "firebrick"]
    names  = [f"0–{epoch_basis//1000}K", "extended"]

    for comp in range(n_comp):
        for mask, color, name in zip([pre_mask, post_mask], colors, names):
            idx = [i for i, m in enumerate(mask) if m]
            if not idx:
                continue
            fig.add_trace(
                go.Scatter(
                    x=[epochs_all[i] for i in idx],
                    y=coords_all[idx, comp].tolist(),
                    mode="lines",
                    name=name,
                    line=dict(color=color, width=1.5),
                    showlegend=(comp == 0),
                ),
                row=comp + 1, col=1,
            )
        # vertical divider at basis epoch
        fig.add_vline(
            x=epoch_basis, line_dash="dash", line_color="gray",
            line_width=1, row=comp + 1, col=1,
        )

    fig.update_xaxes(title_text="epoch", row=n_comp, col=1)
    fig.update_layout(
        title=f"{label} | W_in trajectory PCs vs epoch",
        template="plotly_white",
        height=700,
        legend=dict(x=1.01, y=1),
    )
    return fig

In [20]:
for prime, model_seed, data_seed, label in EXTENDED_MODELS:
    v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)
    fig = plot_trajectory_timeseries(v, label=label)
    fig.show()

---
## 2. Within-Group Parameter Trajectory PCA

**Motivation**: does each frequency group's subset of W_in neurons follow the same
macro path as the global MLP trajectory, but with *different deflection timing*?

**Method**:
1. Fit the global W_in trajectory PCA (basis from all available epochs).
2. For each frequency group, zero out non-member neurons in W_in at every epoch,
   then project through the *global* basis.  
   This extracts each group's *contribution* to the global PC directions.
3. Plot the resulting PC1/PC2 time series overlaid — one trace per group.

If deflection timing differs across groups, their PC1 curves will shift left/right
relative to each other. If they track in lockstep, groups move as a unit.

In [21]:
def _freq_color(freq_idx: int, n_freq: int) -> str:
    hue = freq_idx / max(n_freq, 1)
    r, g, b = colorsys.hls_to_rgb(hue, 0.50, 0.65)
    return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"


def load_group_trajectory_data(variant, epoch_basis=None):
    """Compute group-contribution trajectories projected through the global W_in PCA basis.

    For each frequency group g (neurons indexed by group_members[g]):
        - Construct a sparse W_in where only group g's columns are non-zero.
        - Flatten and project through the global basis.

    This gives each group's independent contribution to the global PC motion.

    Args:
        variant:      Variant object
        epoch_basis:  if set, fit basis only on epochs <= this value;
                      all epochs are still projected (same as Approach B above)

    Returns dict:
        epochs:         list[int]
        global_coords:  (n_epochs, 3)  — global trajectory
        group_coords:   dict[g_idx -> (n_epochs, 3)]  — per-group contribution
        group_freqs:    (n_groups,)
        var_explained:  (3,)
        basis:          (3, d_params)
        d_model, d_mlp: int
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    group_data = loader.load_cross_epoch("neuron_group_pca")
    neuron_group_idx = group_data["neuron_group_idx"]  # (d_mlp,)
    group_freqs = group_data["group_freqs"]             # (n_groups,)

    # full trajectory
    epochs_all, traj_all = load_win_trajectory(variant)

    # fit basis
    if epoch_basis is not None:
        epochs_b, traj_b = load_win_trajectory(variant, epoch_limit=epoch_basis)
    else:
        epochs_b, traj_b = epochs_all, traj_all

    _, basis, center, var_exp = fit_trajectory_pca(traj_b)
    global_coords = project_trajectory(traj_all, basis, center)

    # group contribution: isolate each group's columns in W_in, project through global basis
    snap_shape = loader.load_epoch("parameter_snapshot", epochs_all[0])["W_in"].shape
    d_model, d_mlp = snap_shape  # (d_model, d_mlp)

    group_coords = {}
    for g_idx, freq in enumerate(group_freqs):
        members = np.where(neuron_group_idx == g_idx)[0]
        if len(members) == 0:
            continue

        # Build column-mask: True for group-member neurons
        col_mask = np.zeros(d_mlp, dtype=bool)
        col_mask[members] = True

        # For each epoch, reshape flattened W_in to (d_model, d_mlp), zero non-members, re-flatten
        group_traj = traj_all.reshape(-1, d_model, d_mlp).copy()
        group_traj[:, :, ~col_mask] = 0
        group_traj = group_traj.reshape(-1, d_model * d_mlp)

        # Use the global centering (subtract the full center, then zero non-group dims)
        # This preserves relative position in the global space.
        X_group = (group_traj - center)
        group_coords[g_idx] = X_group @ basis.T   # (n_epochs, 3)

    return {
        "epochs": epochs_all,
        "global_coords": global_coords,
        "group_coords": group_coords,
        "group_freqs": group_freqs,
        "var_explained": var_exp,
        "basis": basis,
        "d_model": d_model,
        "d_mlp": d_mlp,
    }

In [22]:
def plot_group_trajectory_timeseries(variant, label, epoch_basis=None) -> go.Figure:
    """PC1, PC2, PC3 vs epoch: global trajectory + per-group contributions overlaid.

    A vertical dashed line is drawn at `epoch_basis` when set.
    """
    data = load_group_trajectory_data(variant, epoch_basis=epoch_basis)
    epochs = data["epochs"]
    global_coords = data["global_coords"]
    group_coords = data["group_coords"]
    group_freqs = data["group_freqs"]
    var_exp = data["var_explained"]

    n_freq = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1
    prime = int(variant.model_config["prime"])
    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=axis_labels,
    )

    # global trajectory — faint dotted reference
    for comp, row in enumerate([1, 2, 3]):
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=global_coords[:, comp].tolist(),
                mode="lines",
                name="global W_in",
                line=dict(color="rgba(0,0,0,0.20)", width=2, dash="dot"),
                showlegend=(comp == 0),
            ),
            row=row, col=1,
        )

    # per-group contributions
    for g_idx, freq in enumerate(group_freqs):
        if g_idx not in group_coords:
            continue
        coords_g = group_coords[g_idx]
        color = _freq_color(int(freq), n_freq)

        for comp, row in enumerate([1, 2, 3]):
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=coords_g[:, comp].tolist(),
                    mode="lines",
                    name=f"freq {int(freq)+1}",
                    line=dict(color=color, width=2),
                    showlegend=(comp == 0),
                ),
                row=row, col=1,
            )

    if epoch_basis is not None:
        for row in [1, 2, 3]:
            fig.add_vline(
                x=epoch_basis, line_dash="dash", line_color="gray",
                line_width=1, row=row, col=1,
            )

    fig.update_xaxes(title_text="epoch", row=3, col=1)
    fig.update_layout(
        title=f"{label} | p={prime} | per-group W_in trajectory contribution (global basis)",
        template="plotly_white",
        height=750,
        legend=dict(x=1.01, y=1),
    )
    return fig


def plot_group_trajectory_scatter(variant, label, epoch_basis=None) -> go.Figure:
    """PC1 vs PC2 path: global trajectory (dotted) + per-group (colored), all in the same space.

    Epochs are encoded as marker size (larger = later) for legibility.
    """
    data = load_group_trajectory_data(variant, epoch_basis=epoch_basis)
    epochs = data["epochs"]
    global_coords = data["global_coords"]
    group_coords = data["group_coords"]
    group_freqs = data["group_freqs"]
    var_exp = data["var_explained"]
    n_freq = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1
    prime = int(variant.model_config["prime"])

    col_scale = _epoch_colorscale(epochs)
    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]

    fig = go.Figure()

    # global trajectory
    fig.add_trace(go.Scatter(
        x=global_coords[:, 0].tolist(),
        y=global_coords[:, 1].tolist(),
        mode="lines",
        name="global W_in",
        line=dict(color="rgba(0,0,0,0.15)", width=2, dash="dot"),
    ))

    for g_idx, freq in enumerate(group_freqs):
        if g_idx not in group_coords:
            continue
        coords_g = group_coords[g_idx]
        color = _freq_color(int(freq), n_freq)
        fig.add_trace(go.Scatter(
            x=coords_g[:, 0].tolist(),
            y=coords_g[:, 1].tolist(),
            mode="lines+markers",
            name=f"freq {int(freq)+1}",
            marker=dict(
                size=4,
                color=col_scale,
                colorscale="RdYlBu_r",
                cmin=0, cmax=1,
            ),
            line=dict(color=color, width=1.5),
            customdata=epochs,
            hovertemplate="freq %{meta} | epoch %{customdata}<br>PC1=%{x:.3f} PC2=%{y:.3f}<extra></extra>",
            meta=int(freq) + 1,
        ))

    fig.update_layout(
        title=f"{label} | p={prime} | group trajectory contributions in global PC space",
        xaxis_title=axis_labels[0],
        yaxis_title=axis_labels[1],
        template="plotly_white",
        height=550,
        legend=dict(x=1.01, y=1),
    )
    return fig

In [23]:
# ---- Use an extended-training variant (p113/s999/ds42) as the primary example ----
#prime, model_seed, data_seed, label = (113, 999, 42, "p113/s999/ds42")
#prime, model_seed, data_seed, label = (109, 485, 598, "p109/s485/ds598")
#prime, model_seed, data_seed, label = (113, 999, 598, "p113/s999/ds598")
prime, model_seed, data_seed, label = (101, 485, 42, "p101/s485/ds42")
v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)

# Time-series view: PC1 and PC2 vs epoch, global + per-group
fig_ts = plot_group_trajectory_timeseries(v, label=label, epoch_basis=EPOCH_BASIS)
fig_ts.show()

# Scatter view: group contributions in PC1 vs PC2 space
fig_sc = plot_group_trajectory_scatter(v, label=label, epoch_basis=EPOCH_BASIS)
fig_sc.show()

In [24]:
# ---- Compare across variants for robustness ----
COMPARE_MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (101, 999, 598, "p101 open-loop"),
]

for prime, model_seed, data_seed, label in COMPARE_MODELS:
    v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)
    fig = plot_group_trajectory_timeseries(v, label=label)
    fig.show()

In [25]:
for comp, row in enumerate([1, 2]):
    print(f"comp: {comp}, row:{row}")

comp: 0, row:1
comp: 1, row:2


---
## 3. Group Centroid Trajectory PCA

**What this shows:** For each frequency group, the centroid of its member neurons
in W_in column space (a 128-dim vector) is computed at every epoch.
A shared PCA is fit on all group centroids across all epochs — all groups have equal
weight in determining the basis.  Each group's centroid sequence is then projected
through this shared basis, producing directly comparable paths in the same PC space.

**Two versions side by side:**

- **Raw** — coordinates as projected. Reveals which group's centroid moves the
  most (large displacement = dominates the basis). Groups that barely move will
  appear clustered near the origin.

- **Per-group normalized** — each group's trajectory is z-scored across time
  (subtract its own mean, divide by its own std) before display. Equalized scale
  isolates *shape* of movement from *magnitude*. Sigmoid vs. linear rise becomes
  directly comparable across groups.

**Why this fills the gap the animated centroid scatter left:** The animated view
uses group-local coordinate systems (each group's own final-epoch PCA basis),
so paths cannot be compared across groups. Here the basis is shared — all groups
live in the same coordinate frame, and the full arc is readable statically.

In [ ]:
def compute_group_centroids(variant, epoch_limit=None):
    """Compute the W_in centroid of each frequency group at every epoch.

    Handles both transformer-style W_in (d_model, d_mlp) where neurons are
    columns, and MLP-style W_in (d_hidden, d_input) where neurons are rows.
    Architecture is detected by matching neuron_group_idx length to each axis.

    Returns:
        epochs:      list[int]
        centroids:   (n_groups, n_epochs, d_embed)   — d_embed is the non-neuron axis
        group_freqs: (n_groups,)
        d_embed:     int  — centroid dimensionality
        n_neurons:   int  — number of neurons
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    group_data = loader.load_cross_epoch("neuron_group_pca")
    neuron_group_idx = group_data["neuron_group_idx"]   # (n_neurons,)
    group_freqs      = group_data["group_freqs"]         # (n_groups,)
    n_groups  = len(group_freqs)
    n_neurons = len(neuron_group_idx)

    epochs, traj = load_win_trajectory(variant, epoch_limit=epoch_limit)

    snap_shape = loader.load_epoch("parameter_snapshot", epochs[0])["W_in"].shape
    d_first, d_second = snap_shape

    # Detect neuron axis: whichever dimension equals n_neurons
    if d_second == n_neurons:
        neuron_axis = 1          # (d_embed, n_neurons) — transformer style
        d_embed = d_first
    elif d_first == n_neurons:
        neuron_axis = 0          # (n_neurons, d_embed) — MLP style
        d_embed = d_second
    else:
        raise ValueError(
            f"Cannot detect neuron axis: W_in {snap_shape}, n_neurons={n_neurons}"
        )

    W_epochs = traj.reshape(len(epochs), d_first, d_second)  # (n_epochs, d_first, d_second)

    centroids = np.zeros((n_groups, len(epochs), d_embed))
    for g_idx in range(n_groups):
        members = np.where(neuron_group_idx == g_idx)[0]
        if len(members) == 0:
            continue
        if neuron_axis == 1:
            centroids[g_idx] = W_epochs[:, :, members].mean(axis=2)   # (n_epochs, d_embed)
        else:
            centroids[g_idx] = W_epochs[:, members, :].mean(axis=1)   # (n_epochs, d_embed)

    return epochs, centroids, group_freqs, d_embed, n_neurons


def fit_centroid_pca(centroids, n_components=3):
    """Shared PCA on all group centroids across all epochs.

    Each centroid is one row; all groups contribute equally.

    Args:
        centroids: (n_groups, n_epochs, d_model)

    Returns:
        coords:       (n_groups, n_epochs, n_components) — raw projections
        basis:        (n_components, d_model)
        center:       (d_model,)
        var_explained (n_components,)
    """
    n_groups, n_epochs, d_model = centroids.shape
    stacked = centroids.reshape(-1, d_model)             # (n_groups*n_epochs, d_model)
    center  = stacked.mean(axis=0)
    X = stacked - center
    _, S, Vt = np.linalg.svd(X, full_matrices=False)
    basis        = Vt[:n_components]                     # (n_components, d_model)
    var_explained = (S**2 / (S**2).sum())[:n_components]

    # Project: (n_groups, n_epochs, d_model) @ (d_model, n_components)
    coords = (centroids - center) @ basis.T              # (n_groups, n_epochs, n_components)
    return coords, basis, center, var_explained


def normalize_per_group(coords):
    """Z-score each group's trajectory independently (axis=1 = epoch axis).

    Subtracts each group's temporal mean and divides by its temporal std.
    Equates scale across groups so shape of movement is directly comparable.

    Args:
        coords: (n_groups, n_epochs, n_components)

    Returns:
        normalized: (n_groups, n_epochs, n_components)
    """
    mean = coords.mean(axis=1, keepdims=True)            # (n_groups, 1, n_components)
    std  = coords.std(axis=1,  keepdims=True).clip(1e-8)
    return (coords - mean) / std

In [27]:
def _centroid_path_traces(coords, group_freqs, epochs, col_scale, n_freq, show_legend):
    """Build one Scatter trace per group for a single PC plane.

    Line color = group's frequency color (who it is).
    Marker color = epoch (when it was there).
    """
    traces = []
    for g_idx, freq in enumerate(group_freqs):
        color = _freq_color(int(freq), n_freq)
        traces.append(go.Scatter(
            x=coords[g_idx, :, 0].tolist(),
            y=coords[g_idx, :, 1].tolist(),
            mode="lines+markers",
            name=f"freq {int(freq)+1}",
            line=dict(color=color, width=2),
            marker=dict(
                size=5,
                color=col_scale,
                colorscale="RdYlBu_r",
                cmin=0, cmax=1,
                showscale=False,
            ),
            customdata=epochs,
            hovertemplate=(
                f"freq {int(freq)+1}<br>"
                "epoch %{customdata}<br>"
                "x=%{x:.3f}  y=%{y:.3f}<extra></extra>"
            ),
            showlegend=show_legend,
        ))
    return traces


def plot_centroid_trajectory(variant, label, epoch_limit=None) -> go.Figure:
    """2×2 grid: PC1vPC2 and PC2vPC3, raw vs per-group normalized.

    Row 1 — PC1 vs PC2
    Row 2 — PC2 vs PC3
    Col 1 — raw projections  (reveals which group dominates by displacement)
    Col 2 — per-group normalized  (z-scored per group; isolates shape of movement)

    Line color encodes frequency group identity.
    Marker color encodes epoch (blue=early, red=late).
    """
    epochs, centroids, group_freqs, d_model, d_mlp = compute_group_centroids(
        variant, epoch_limit=epoch_limit
    )
    coords_raw, basis, center, var_exp = fit_centroid_pca(centroids)
    coords_norm = normalize_per_group(coords_raw)

    n_freq    = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1
    col_scale = _epoch_colorscale(epochs)
    prime     = int(variant.model_config["prime"])

    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]

    # Subtitle note: variance explained is from the raw fit;
    # normalized coords are rescaled so percentages no longer apply.
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f"PC1 vs PC2 — raw",
            f"PC1 vs PC2 — normalized (shape only)",
            f"PC2 vs PC3 — raw",
            f"PC2 vs PC3 — normalized (shape only)",
        ],
        horizontal_spacing=0.12,
        vertical_spacing=0.14,
    )

    planes = [
        (0, 1, 1),   # PC1 vs PC2, row 1
        (1, 2, 2),   # PC2 vs PC3, row 2
    ]

    for x_comp, y_comp, plot_row in planes:
        x_lbl_raw  = axis_labels[x_comp]
        y_lbl_raw  = axis_labels[y_comp]
        x_lbl_norm = f"PC{x_comp+1} (normalized)"
        y_lbl_norm = f"PC{y_comp+1} (normalized)"

        for col, (coords, x_lbl, y_lbl) in enumerate(
            [(coords_raw, x_lbl_raw, y_lbl_raw),
             (coords_norm, x_lbl_norm, y_lbl_norm)],
            start=1,
        ):
            # Slice to the two components we're plotting
            coords_plane = coords[:, :, [x_comp, y_comp]]  # (n_groups, n_epochs, 2)

            show_legend = (plot_row == 1 and col == 1)
            for trace in _centroid_path_traces(
                coords_plane, group_freqs, epochs, col_scale, n_freq, show_legend
            ):
                fig.add_trace(trace, row=plot_row, col=col)

            fig.update_xaxes(title_text=x_lbl, row=plot_row, col=col)
            fig.update_yaxes(title_text=y_lbl, row=plot_row, col=col)

    # Invisible colorbar trace to show epoch scale
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(
            colorscale="RdYlBu_r", cmin=0, cmax=1,
            color=[0], showscale=True,
            colorbar=dict(
                title="epoch",
                tickvals=[0, 0.5, 1],
                ticktext=[str(epochs[0]), str(epochs[len(epochs)//2]), str(epochs[-1])],
                len=0.45, x=1.02, y=0.22,
            ),
        ),
        showlegend=False,
    ), row=2, col=2)

    fig.update_layout(
        title=f"{label} | p={prime} | group centroid trajectory PCA (shared basis, d_model={d_model})",
        template="plotly_white",
        height=900,
        legend=dict(x=1.01, y=1),
    )
    return fig

In [28]:
# ---- Primary comparison: canon (diffuse) vs reference healthy vs open-loop ----
CENTROID_MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (101, 999, 598, "p101 open-loop"),
    (113, 999, 42,  "p113/s999/ds42 (extended)"),
]

for prime, model_seed, data_seed, label in CENTROID_MODELS:
    v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)
    fig = plot_centroid_trajectory(v, label=label)
    fig.show()

### 3b. Per-group local trajectory PCA — unfolding the arcs

The shared basis in Section 3a gives one fixed viewing angle onto all groups.
If each group traces the same arc in its own subspace of d_model space, each group
will appear as a *different projection* of the same underlying shape — some face-on,
some edge-on, some at intermediate angles.

**Test:** fit PCA on each group's centroid sequence *independently*.
This finds the orientation that best represents *that group's* arc, effectively
"unfolding" it flat before we look at it.

- **Top row** — each group in its own local PCA frame, colored by epoch.
  If they're all the same shape, they'll all look like the same arc.

- **Overlay** — all groups normalized to unit scale and overlaid in one plot.
  Tight agreement = same shape up to scale and reflection.

**Procrustes pairwise distances** provide the rigorous scalar summary:
given two trajectories, find the rotation + reflection that best aligns them
and measure the residual. Small disparity = same shape.

In [29]:
from scipy.spatial import procrustes as scipy_procrustes


def fit_per_group_local_pca(centroids, n_components=3):
    """PCA fit independently on each group's centroid trajectory.

    Args:
        centroids: (n_groups, n_epochs, d_model)

    Returns:
        local_coords:   list[np.ndarray (n_epochs, n_components)]
        local_var_exp:  list[np.ndarray (n_components,)]
    """
    local_coords  = []
    local_var_exp = []
    for g in range(centroids.shape[0]):
        X = centroids[g]                    # (n_epochs, d_model)
        center = X.mean(axis=0)
        X_c = X - center
        _, S, Vt = np.linalg.svd(X_c, full_matrices=False)
        basis = Vt[:n_components]
        local_coords.append(X_c @ basis.T)  # (n_epochs, n_components)
        local_var_exp.append((S**2 / (S**2).sum())[:n_components])
    return local_coords, local_var_exp


def normalize_arc(coords_2d):
    """Normalize a 2D trajectory to unit range along its dominant axis (PC1).

    Divides by the peak-to-peak range of PC1 so scale is comparable across groups.
    Sign is fixed so PC1 always runs left-to-right (first epoch at lower PC1).
    """
    coords = coords_2d.copy()
    if coords[-1, 0] < coords[0, 0]:    # flip so arc reads left-to-right
        coords *= -1
    pc1_range = np.ptp(coords[:, 0])
    if pc1_range > 1e-8:
        coords = coords / pc1_range
    return coords


def compute_procrustes_matrix(local_coords_2d):
    """Pairwise Procrustes disparity between groups' 2D local trajectory arcs.

    scipy.spatial.procrustes standardizes each trajectory to zero mean and unit
    Frobenius norm, then finds the orthogonal transformation that minimises the
    sum of squared point distances.  Disparity ∈ [0, 1]: 0 = identical shape.

    Args:
        local_coords_2d: list of (n_epochs, 2)

    Returns:
        disparity_matrix: (n_groups, n_groups) symmetric, zeros on diagonal
    """
    n = len(local_coords_2d)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            try:
                _, _, disp = scipy_procrustes(local_coords_2d[i], local_coords_2d[j])
            except Exception:
                disp = np.nan
            D[i, j] = D[j, i] = disp
    return D

In [ ]:
MAX_GRID_GROUPS = 8   # beyond this, skip per-group row and go straight to overlay + heatmap


def plot_arc_unfolding(variant, label, epoch_limit=None) -> go.Figure:
    """Per-group local PCA arcs + normalized overlay + Procrustes heatmap.

    When n_groups <= MAX_GRID_GROUPS:
        Row 1, cols 1..n_groups  — individual arcs in local PCA frame
        Row 2, col 1             — all arcs normalized and overlaid
        Row 2, col 2             — Procrustes pairwise disparity heatmap

    When n_groups > MAX_GRID_GROUPS (e.g. 2L MLP with 56 groups):
        Row 1, col 1             — normalized overlay  (no per-group grid)
        Row 1, col 2             — Procrustes pairwise disparity heatmap
    """
    epochs, centroids, group_freqs, d_embed, n_neurons = compute_group_centroids(
        variant, epoch_limit=epoch_limit
    )
    local_coords, local_var_exp = fit_per_group_local_pca(centroids, n_components=3)

    n_groups  = len(group_freqs)
    n_freq    = int(group_freqs.max()) + 1 if n_groups > 0 else 1
    col_scale = _epoch_colorscale(epochs)
    prime     = int(variant.model_config["prime"])

    freq_colors  = [_freq_color(int(f), n_freq) for f in group_freqs]
    group_labels = [f"freq {int(f)+1}" for f in group_freqs]

    # Print Procrustes table regardless of layout
    D = compute_procrustes_matrix([c[:, :2] for c in local_coords])
    print(f"\nProcrustes pairwise disparity — {label}")
    if n_groups <= 8:
        header = "         " + "  ".join(f"{lbl:>10}" for lbl in group_labels)
        print(header)
        for i, row_lbl in enumerate(group_labels):
            vals = "  ".join(
                f"{D[i,j]:>10.4f}" if i != j else f"{'—':>10}"
                for j in range(n_groups)
            )
            print(f"{row_lbl:>8}  {vals}")
    else:
        off_diag = D[D > 0].flatten()
        print(f"  {n_groups} groups — summary: min={off_diag.min():.4f}  "
              f"median={float(np.median(off_diag)):.4f}  max={off_diag.max():.4f}")

    show_grid = n_groups <= MAX_GRID_GROUPS

    if show_grid:
        n_cols = max(n_groups, 2)
        fig = make_subplots(
            rows=2, cols=n_cols,
            subplot_titles=(
                [f"freq {int(f)+1}  (PC1={v[0]:.0%}, PC2={v[1]:.0%})"
                 for f, v in zip(group_freqs, local_var_exp)]
                + ["normalized overlay", "Procrustes disparity"]
                + [""] * max(0, n_cols - 2)
            ),
            horizontal_spacing=max(0.04, 0.9 / max(n_cols - 1, 1)),
            vertical_spacing=0.18,
        )
        for g_idx, (coords, color, lbl) in enumerate(
            zip(local_coords, freq_colors, group_labels), start=1
        ):
            fig.add_trace(
                go.Scatter(
                    x=coords[:, 0].tolist(), y=coords[:, 1].tolist(),
                    mode="lines+markers", name=lbl,
                    line=dict(color=color, width=2),
                    marker=dict(size=5, color=col_scale, colorscale="RdYlBu_r",
                                cmin=0, cmax=1, showscale=False),
                    customdata=epochs,
                    hovertemplate=(f"{lbl}<br>epoch %{{customdata}}<br>"
                                   "PC1=%{x:.3f} PC2=%{y:.3f}<extra></extra>"),
                    showlegend=False,
                ),
                row=1, col=g_idx,
            )
            fig.update_xaxes(title_text="local PC1", row=1, col=g_idx)
            fig.update_yaxes(title_text="local PC2", row=1, col=g_idx)
        overlay_row, overlay_col = 2, 1
        heatmap_row, heatmap_col = 2, 2
    else:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=["normalized overlay", "Procrustes disparity"],
            horizontal_spacing=0.12,
        )
        overlay_row, overlay_col = 1, 1
        heatmap_row, heatmap_col = 1, 2

    # Normalized overlay
    normed = [normalize_arc(c[:, :2]) for c in local_coords]
    for coords_n, color, lbl in zip(normed, freq_colors, group_labels):
        fig.add_trace(
            go.Scatter(
                x=coords_n[:, 0].tolist(), y=coords_n[:, 1].tolist(),
                mode="lines+markers", name=lbl,
                line=dict(color=color, width=1.5 if n_groups > MAX_GRID_GROUPS else 2),
                marker=dict(size=4 if n_groups > MAX_GRID_GROUPS else 5,
                            color=col_scale, colorscale="RdYlBu_r",
                            cmin=0, cmax=1, showscale=False),
                customdata=epochs,
                hovertemplate=(f"{lbl}<br>epoch %{{customdata}}<br>"
                               "x=%{x:.3f} y=%{y:.3f}<extra></extra>"),
                showlegend=(n_groups <= MAX_GRID_GROUPS),
            ),
            row=overlay_row, col=overlay_col,
        )
    fig.update_xaxes(title_text="PC1 (normalized)", row=overlay_row, col=overlay_col)
    fig.update_yaxes(title_text="PC2 (normalized)", row=overlay_row, col=overlay_col)

    # Epoch colorbar dummy trace
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(
            colorscale="RdYlBu_r", cmin=0, cmax=1, color=[0], showscale=True,
            colorbar=dict(
                title="epoch",
                tickvals=[0, 0.5, 1],
                ticktext=[str(epochs[0]), str(epochs[len(epochs)//2]), str(epochs[-1])],
                len=0.38, x=1.14, y=0.20,
            ),
        ),
        showlegend=False,
    ), row=overlay_row, col=overlay_col)

    # Procrustes heatmap
    fig.add_trace(
        go.Heatmap(
            z=D.tolist(), x=group_labels, y=group_labels,
            colorscale="RdYlGn_r", zmin=0, zmax=0.5,
            showscale=True,
            colorbar=dict(
                title="disparity",
                tickvals=[0, 0.25, 0.5],
                ticktext=["0 (identical)", "0.25", "≥0.5 (unrelated)"],
                len=0.38, x=1.02, y=0.20,
            ),
            hovertemplate=("<b>%{y}</b> vs <b>%{x}</b><br>"
                           "Procrustes disparity: %{z:.4f}<extra></extra>"),
        ),
        row=heatmap_row, col=heatmap_col,
    )

    height = 850 if show_grid else 500
    fig.update_layout(
        title=f"{label} | p={prime} | arc unfolding (d_embed={d_embed}, {n_groups} groups)",
        template="plotly_white",
        height=height,
        legend=dict(x=1.01, y=0.55),
    )
    return fig

In [32]:
# ---- p109: the reference healthy model, where the structure should be clearest ----
v_p109 = family.get_variant(prime=109, seed=485, data_seed=598)
fig = plot_arc_unfolding(v_p109, label="p109 reference healthy")
fig.show()


Procrustes pairwise disparity — p109 reference healthy
             freq 4     freq 14     freq 27
  freq 4           —      0.0356      0.0846
 freq 14      0.0356           —      0.0610
 freq 27      0.0846      0.0610           —


In [33]:
# ---- Compare against pathological models ----
# If the factored structure is specific to healthy grokking, p101 and p113 should show
# higher Procrustes disparity and less agreement in the overlay.
for prime, model_seed, data_seed, label in [
    (113, 999, 598, "p113 canon (diffuse)"),
    (101, 999, 598, "p101 open-loop"),
]:
    v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)
    fig = plot_arc_unfolding(v, label=label)
    fig.show()


Procrustes pairwise disparity — p113 canon (diffuse)
             freq 9     freq 33     freq 38     freq 55
  freq 9           —      0.0350      0.0685      0.0567
 freq 33      0.0350           —      0.0961      0.1244
 freq 38      0.0685      0.0961           —      0.0820
 freq 55      0.0567      0.1244      0.0820           —



Procrustes pairwise disparity — p101 open-loop
            freq 35     freq 41     freq 43     freq 44
 freq 35           —      0.0532      0.0098      0.1411
 freq 41      0.0532           —      0.0308      0.1604
 freq 43      0.0098      0.0308           —      0.1504
 freq 44      0.1411      0.1604      0.1504           —


---
## 4. Architecture ladder comparison

Same arc analysis run on the **2-layer MLP** (p=113 and p=97, full analyzer coverage)
and noted gap on the **learned-embedding MLP** (no `neuron_group_pca` yet).

**Caveats before reading:**
- 2L MLP has 56 frequency groups with only 3–11 neurons each (vs. 3–4 groups
  with ~150 neurons in the transformer). Centroids are noisier.
- Only 2 trained variants available. Cross-variant robustness is limited.
- Learned-emb MLP excluded: `neuron_group_pca` not yet run — that is the
  analyzer gap that a new REQ should address.

Early signal only. Treat as directional, not conclusive.

In [ ]:
# ---- 2L MLP: shared centroid trajectory PCA ----
fam_2l = load_family('modulo_addition_2layer_mlp')

for v in fam_2l.list_variants():
    p = v.params
    label = f"2L-MLP p={p['prime']}/s={p['seed']}/ds={p['data_seed']}"
    print(f"\n{label}")
    fig = plot_centroid_trajectory(v, label=label)
    fig.show()

In [ ]:
# ---- 2L MLP: arc unfolding ----
# 56 groups makes the per-group grid impractical; the Procrustes heatmap
# is the readable output. If most of the 56x56 matrix is green, the
# arc-similarity hypothesis extends to the 2L MLP.
for v in fam_2l.list_variants():
    p = v.params
    label = f"2L-MLP p={p['prime']}/s={p['seed']}/ds={p['data_seed']}"
    # Arc unfolding will print the full Procrustes matrix to stdout
    # and show the heatmap. With 56 groups the grid row is wide — zoom out.
    fig = plot_arc_unfolding(v, label=label)
    fig.show()

In [ ]:
# ---- Learned-emb MLP: analyzer gap ----
# neuron_group_pca has not been run for this family.
# Available analyzers: effective_dimensionality, parameter_snapshot,
# neuron_freq_norm, repr_geometry, neuron_activations.
#
# To run arc analysis here, the pipeline needs:
#   1. neuron_group_pca run and committed
#   2. Ideally: more trained variants (currently only p=113/s=999/ds=598)
#
# This is the REQ scope: extend analyzer coverage to learned-emb MLP
# so the ladder comparison is on equal footing.

fam_emb = load_family('modulo_addition_learned_emb_mlp')
v_emb = fam_emb.list_variants()[0]
from miscope.analysis.artifact_loader import ArtifactLoader as _AL
loader_emb = _AL(str(v_emb.variant_dir / 'artifacts'))
print('Learned-emb MLP available analyzers:')
import os
for d in sorted(os.listdir(v_emb.variant_dir / 'artifacts')):
    if os.path.isdir(v_emb.variant_dir / 'artifacts' / d):
        print(f'  {d}')
print('\nneuron_group_pca: NOT available — arc analysis cannot run')